# Semantic Search & LLM Application
**Starter Notebook**

Notebook ini adalah kerangka awal untuk membangun prototype aplikasi semantic search yang terintegrasi dengan LLM, menggunakan knowledge base FAQ e-commerce.

Alur besar: Load Knowledge Base → Embedding → Vector Database (FAISS) → Semantic Search → Prompt Design → LLM Integration → Uji Relevansi.

Isi tiap section sesuai instruksi di Assignment Brief. Jangan ubah struktur section, tapi silakan tambah cell baru di dalam tiap section jika diperlukan.

## 1. Import Library & Load Knowledge Base

### 1a.Using Google API

In [ ]:
import pandas as pd
import numpy as np
import faiss
from google import genai  # sesuaikan dengan library Gemini API yang digunakan
from google.genai import types
import os

client = genai.Client(api_key=os.environ.get('GEMINI_API_KEY'))

df = pd.read_csv('../data/ecommerce_faq_knowledge_base.csv')
print('Total entri knowledge base:', len(df))
df.head()

### 1b. Using Openrouter

In [1]:
import pandas as pd
import numpy as np
import faiss
from google import genai  # sesuaikan dengan library Gemini API yang digunakan
from google.genai import types
import os
import openai
from dotenv import load_dotenv
import requests

load_dotenv()

client_open_router =  os.environ.get("API_KEY_OPENROUTER")

df = pd.read_csv('../data/ecommerce_faq_knowledge_base.csv')
print('Total entri knowledge base:', len(df))
df.head()

Total entri knowledge base: 59


,faq_id,category,question,answer
0,1,Pemesanan,Bagaimana cara melakukan pemesanan di platform...,"Pilih produk yang diinginkan, klik 'Tambah ke ..."
1,2,Pemesanan,Apakah saya bisa memesan tanpa membuat akun?,Saat ini pemesanan hanya bisa dilakukan setela...
2,3,Pemesanan,Bagaimana cara membatalkan pesanan yang sudah ...,Pesanan dapat dibatalkan melalui halaman 'Pesa...
3,4,Pemesanan,Berapa lama waktu yang dibutuhkan sebelum pesa...,Penjual biasanya memproses pesanan dalam waktu...
4,5,Pemesanan,Apakah saya bisa mengubah alamat pengiriman se...,Perubahan alamat hanya bisa dilakukan sebelum ...


## 2. Menyiapkan Unit Retrievable

Setiap entri FAQ (question + answer) diperlakukan sebagai satu unit yang akan di-embed dan disimpan ke vector database. Gabungkan question dan answer menjadi satu teks representatif per entri.

In [3]:
df['content'] = 'Q: ' + df['question'] + ' A: ' + df['answer']
documents = df['content'].tolist()
print('Contoh unit retrievable:')
print(documents[0])

Contoh unit retrievable:
Q: Bagaimana cara melakukan pemesanan di platform ini? A: Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran.


## 3. Membuat Embeddings

Ubah setiap unit retrievable menjadi vektor menggunakan Gemini Embeddings API.

### 3a. Membuat Embeddings - Model via Google API

In [ ]:
def get_embedding(text):
    result = client.models.embed_content(
        model='text-embedding-004',  # sesuaikan versi model embedding
        contents=text
    )
    return result.embeddings[0].values

embeddings = [get_embedding(doc) for doc in documents]
embeddings = np.array(embeddings).astype('float32')
print('Shape embeddings:', embeddings.shape)

### 3b. Membuat Embeddings - Model via Openroute

In [19]:
def get_embedding_openrouter(text):
    result= requests.post(
        "https://openrouter.ai/api/v1/embeddings",
        headers={
            "Authorization": f"Bearer {client_open_router}",
            "Content-Type": "application/json",
        },
        json={"model": "google/gemini-embedding-001",
            "input": f"{text}",}
    )
    data = result.json()
    return data["data"][0]["embedding"]

embeddings = [get_embedding_openrouter(doc) for doc in documents]
embeddings = np.array(embeddings).astype('float32')
print('Shape embeddings:', embeddings.shape)


Shape embeddings: (59, 3072)


#### 3ba check embedding result

In [24]:
print(f"{embeddings[0]}")  # Print the first 10 dimensions of the first embedding

[ 0.00192007 -0.00107109  0.01130075 ...  0.00192543  0.01178652
 -0.00955423]


## 4. Menyimpan Embeddings ke Vector Database (FAISS)

In [25]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print('Jumlah vektor tersimpan di index:', index.ntotal)

Jumlah vektor tersimpan di index: 59


## 5. Mengimplementasikan Semantic Search

Fungsi untuk mengambil beberapa entri FAQ paling relevan berdasarkan query pengguna.

In [27]:
def semantic_search(query, top_k=3):
    query_embedding = np.array([get_embedding_openrouter(query)]).astype('float32')
    distances, indices = index.search(query_embedding, top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        results.append({
            'rank': rank + 1,
            'question': df.iloc[idx]['question'],
            'answer': df.iloc[idx]['answer'],
            'category': df.iloc[idx]['category'],
            'distance': distances[0][rank]
        })
    return results

# Contoh percobaan
test_results = semantic_search('kenapa paket saya belum sampai?')
for r in test_results:
    print(r)

{'rank': 1, 'question': 'Bagaimana jika paket saya belum sampai melebihi estimasi?', 'answer': 'Jika paket melebihi estimasi pengiriman lebih dari 3 hari, silakan hubungi customer service dengan menyertakan nomor resi untuk ditelusuri lebih lanjut.', 'category': 'Pengiriman', 'distance': np.float32(0.33353505)}
{'rank': 2, 'question': 'Kenapa status pengiriman saya tidak berubah selama beberapa hari?', 'answer': 'Status yang tidak berubah bisa disebabkan keterlambatan update sistem kurir. Disarankan menunggu 1x24 jam sebelum melaporkan ke customer service.', 'category': 'Pengiriman', 'distance': np.float32(0.40880978)}
{'rank': 3, 'question': 'Kenapa nomor resi saya belum bisa dilacak di website kurir?', 'answer': "Nomor resi baru dapat dilacak setelah paket diserahterimakan ke pihak kurir, biasanya butuh waktu beberapa jam setelah status 'Dikirim'.", 'category': 'Pengiriman', 'distance': np.float32(0.4320181)}


## 6. Mendesain Prompt dan Context

Susun prompt terstruktur (Persona, Task, Context, Format) yang menyisipkan hasil semantic search sebagai bagian Context, sebelum dikirim ke LLM.

In [28]:
def build_prompt(query, search_results):
    context = '\n\n'.join(
        f"FAQ: {r['question']}\nJawaban: {r['answer']}" for r in search_results
    )
    prompt = f'''Kamu adalah asisten customer service e-commerce yang ramah dan membantu.

Tugas: Jawab pertanyaan pengguna HANYA berdasarkan informasi FAQ berikut. Jika informasi tidak ditemukan dalam FAQ, katakan bahwa kamu belum memiliki informasi tersebut dan sarankan menghubungi customer service.

Konteks FAQ relevan:
{context}

Pertanyaan pengguna: {query}

Jawaban:'''
    return prompt

In [40]:
prompt = build_prompt('kenapa paket saya belum sampai?', test_results)
print(prompt)

Kamu adalah asisten customer service e-commerce yang ramah dan membantu.

Tugas: Jawab pertanyaan pengguna HANYA berdasarkan informasi FAQ berikut. Jika informasi tidak ditemukan dalam FAQ, katakan bahwa kamu belum memiliki informasi tersebut dan sarankan menghubungi customer service.

Konteks FAQ relevan:
FAQ: Bagaimana jika paket saya belum sampai melebihi estimasi?
Jawaban: Jika paket melebihi estimasi pengiriman lebih dari 3 hari, silakan hubungi customer service dengan menyertakan nomor resi untuk ditelusuri lebih lanjut.

FAQ: Kenapa status pengiriman saya tidak berubah selama beberapa hari?
Jawaban: Status yang tidak berubah bisa disebabkan keterlambatan update sistem kurir. Disarankan menunggu 1x24 jam sebelum melaporkan ke customer service.

FAQ: Kenapa nomor resi saya belum bisa dilacak di website kurir?
Jawaban: Nomor resi baru dapat dilacak setelah paket diserahterimakan ke pihak kurir, biasanya butuh waktu beberapa jam setelah status 'Dikirim'.

Pertanyaan pengguna: kenapa

## 7. Mengintegrasikan LLM

### 7a Mengintegrasikan LLM - Gemini API

In [ ]:
def generate_answer(query, top_k=3):
    search_results = semantic_search(query, top_k=top_k)
    prompt = build_prompt(query, search_results)

    response = client.models.generate_content(
        model='gemini-2.0-flash',  # sesuaikan versi model
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.3)
    )
    return response.text, search_results

answer, sources = generate_answer('apakah bisa bayar pakai cicilan?')
print('Jawaban LLM:')
print(answer)
print()
print('Sumber FAQ yang digunakan:')
for s in sources:
    print('-', s['question'])

### 7b Mengintegrasikan LLM - Openrouter API

In [60]:
import json
def generate_answer(query, top_k=3):
    search_results = semantic_search(query, top_k=top_k)
    prompt = build_prompt(query, search_results)

    response = requests.post(
        url="https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {client_open_router}",
        },
        data=json.dumps({
            "model": "google/gemini-2.5-flash",
            "messages": [
                {
                    "role": "user",
                    "content": f"{prompt}"
                }
            ],
            "temperature": 0.7
        })
        
    )
    data = response.json()
    return data["choices"][0]["message"]["content"], search_results

answer, sources = generate_answer('apakah bisa bayar pakai cicilan?')
print('Jawaban LLM:')
print(answer)
print()
print('Sumber FAQ yang digunakan:')
for s in sources:
    print('-', s['question'])

Jawaban LLM:
Tentu saja! Fitur cicilan tersedia untuk transaksi dengan kartu kredit tertentu pada nominal minimal yang ditentukan pihak bank.

Sumber FAQ yang digunakan:
- Apakah saya bisa membayar dengan cicilan?
- Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?
- Metode pembayaran apa saja yang tersedia?


## 8. Menguji Relevansi Hasil Retrieval dan Response AI

Uji sistem dengan beberapa query berbeda, termasuk query yang menggunakan kata/frasa berbeda dari FAQ asli (sinonim/parafrase) untuk menilai apakah semantic search benar-benar menangkap makna, bukan hanya mencocokkan kata kunci.

_Uji minimal 5 query berbeda, termasuk:_
- _1-2 query yang jelas ada jawabannya di FAQ_
- _1-2 query dengan kata-kata berbeda dari FAQ asli (parafrase/sinonim)_
- _1 query yang TIDAK ada jawabannya di FAQ (uji bagaimana LLM merespons ketidaktahuannya)_

In [61]:
test_queries = [
    # TODO: isi minimal 5 query uji coba di sini
    "apakah jumlah stock valid?",
    "bagaimana cara mengembalikan barang yang rusak?",
    "apakah ada promo untuk pembelian pertama?",
    "berapa lama waktu pengiriman ke luar negeri?",
    "apakah bisa mengubah alamat pengiriman setelah order?"
]

for q in test_queries:
    print('Query:', q)
    answer, sources = generate_answer(q)
    print('Jawaban:', answer)
    print('Sumber teratas:', sources[0]['question'] if sources else '-')
    print('---')

Query: apakah jumlah stock valid?
Jawaban: Halo! Mengenai jumlah stok, stok di aplikasi kami diperbarui secara berkala oleh penjual. Namun, terkadang bisa terjadi selisih kecil apabila ada transaksi bersamaan dari banyak pembeli.

Semoga informasi ini membantu ya!
Sumber teratas: Apakah stok yang tertera di aplikasi selalu update secara real-time?
---
Query: bagaimana cara mengembalikan barang yang rusak?
Jawaban: Untuk mengembalikan barang yang rusak, Anda perlu segera mengambil foto/video kondisi paket saat diterima dan melaporkannya melalui menu komplain dalam waktu 2x24 jam sejak barang diterima.
Sumber teratas: Bagaimana prosedur pengembalian barang yang tidak sesuai?
---
Query: apakah ada promo untuk pembelian pertama?
Jawaban: Tentu, ada! Pengguna baru biasanya mendapat voucher selamat datang yang dapat digunakan pada transaksi pertama. Silakan cek halaman promo setelah pendaftaran untuk informasi lebih lanjut.
Sumber teratas: Apakah member baru mendapat diskon khusus?
---
Query

### Catatan Analisis Relevansi

_Tuliskan pengamatanmu di sini: query mana yang berhasil di-retrieve dengan baik, query mana yang kurang relevan hasilnya, dan bagaimana LLM merespons ketika informasi tidak tersedia di knowledge base._